# Retrieval-Augmented Generation (RAG)


## 0. Introduction

Like any tool, a large language model (LLM) has limitations. The four that matter most here:

- **Hallucination**: an LLM can produce fluent, confident answers that are incorrect, made up, or not supported by any source.
- **Limited knowledge**: an LLM only knows what was in its training data. It knows nothing about recent events, or about information it has never seen, such as **your private documents**.
- **Limited context size**: an LLM can only process a limited amount of text per call.
- **Unreliable sourcing**: when asked to cite its sources, an LLM may invent them or get them wrong. It cannot cite what it never saw.

**So how can we make an LLM knowledgeable about our documents, and reliable?**

Two obvious options come to mind:

- **Paste all the documents into the prompt.** But:
    - The documents may exceed the **context size**.
    - It is **expensive**: you pay for every input and output token, on every call.
    - **Quality can suffer**: with a lot of unrelated text, the model may overlook a detail (the "lost in the middle" effect). Recent models handle long prompts much better, but the risk remains.
- **Fine-tune (retrain) the model** on our data. But:
    - The model is **frozen** at the moment of training: new or changed documents mean training again.
    - It is **complex**: you must train, host, maintain and deploy your own model.
    - It still gives **no clear, direct sources**.

What if we avoided fine-tuning and still gave the LLM our documents, but **only the relevant parts**? That is exactly what **Retrieval-Augmented Generation (RAG)** does. The name describes its three steps:

- **Retrieval**: find the passages of our documents that are relevant to the question.
- **Augmentation**: add those passages to the prompt.
- **Generation**: the LLM writes the answer, based on that context.

In practice this runs in two phases:
1. **Indexing (offline, prepared in advance)**: split your documents into smaller **chunks** and store them in a database, together with their **embeddings**. This makes the chunks searchable by meaning (a bit like a web search, although the mechanism is different).
2. **Querying (online, for every question)**: use the user's question to search the database for the **most relevant chunks**, paste them into the LLM prompt, and ask the LLM to **answer based on this context only**. ("Top-k" means the *k* best matches, e.g. the 3 most similar chunks.)

Vocabulary:
- **Chunk**: a small piece of a document, typically a section or a few paragraphs, that covers one idea. It is the unit we search for and paste into the prompt.
- **Embedding**: a list of numbers that captures the meaning of a text, so that texts with similar meanings get similar numbers. This lets us search by meaning instead of by exact words.
- **Token**: the unit an LLM reads and is billed by, roughly ¾ of a word. A 9-page document is about 9,000 tokens.

![RAG_scheme](data/figures/RAG_diagram.png)

This way, the LLM can work with our documents without receiving all of them, and without any fine-tuning.

The benefits, among others:

- **Fills the knowledge gap**: the LLM can work with your own documents.
- **Fewer hallucinations**: the context contains the answer, so the model has less reason to invent one (fewer, not zero).
- **Citations**: the LLM can point to the exact passage an answer comes from, and you can even link to it.
- **Focused context**: the model gets the information it needs, without a lot of irrelevant text.
- **Lower cost**: you pay for a few chunks instead of the whole knowledge base.
- **Easy to keep current**: when a document changes, you only update the index, not the model.

*Let's now see how to build it!*

## Setup

Imports, the models and the API client: the API key is read from `.env.local` in the repository root.

In [ ]:
# Setup - can be ignored
import os
import re
import time
from pathlib import Path

import numpy as np
import pypdfium2 as pdfium
from dotenv import load_dotenv
from IPython.display import display
from openai import OpenAI
from pypdf import PdfReader

CHAT_MODEL = "gpt-6-luna"  # generates the answers
EMBEDDING_MODEL = "text-embedding-3-small"  # turns texts into vectors

# walk up from the working directory to the repo root (the folder with .git)
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
load_dotenv(REPO_ROOT / ".env.local")

# An empty LLM_BASE_URL means OpenAI itself
client = OpenAI(api_key=os.environ["LLM_API_KEY"], base_url=os.getenv("LLM_BASE_URL") or None)

## 1. The document

**Our example.** A customer of the health insurer *Groupe Mutuel* has a question about their supplementary health insurance. The answer is in the insurer's *General Terms and Conditions (CGC)*, a 9-page PDF with 47 articles. We want a correct answer that **cites the article** it comes from. The same approach works for many PDFs, but we use a single one to keep things readable.

**The question we want to answer:**

> *Groupe Mutuel just paid out a claim under my supplementary health insurance. Can I cancel this supplementary insurance because of it, and how many days do I have?*

**The relevant part of the document (Art. 13.4):**

> *13.4. After each claim for which a benefit is paid by the insurer, the policyholder has the right to withdraw from the insurance product concerned within 10 days of becoming aware of the payment of the benefit. [...] If the policyholder withdraws from the contract, the insurance ceases to be effective 14 days after the notice of termination has been sent to the insurer.*

**The expected answer:** 

> Yes. After a claim has been paid, the policyholder may withdraw from that insurance **within 10 days** of becoming aware of the payment (Art. 13.4). The insurance then ends 14 days after the notice is sent.

Here is the PDF: [`CAGA04-E2.pdf`](data/insurance_terms/CAGA04-E2.pdf). It is in the folder `data/insurance_terms/`. We display all of its pages below. The passage above is on one of them.

In [ ]:
PDF_PATH = Path("data/insurance_terms/CAGA04-E2.pdf")

pdf = pdfium.PdfDocument(PDF_PATH)
answer_page = next(n for n, page in enumerate(pdf) if "After each claim for which a benefit" in page.get_textpage().get_text_range())
print(f"{len(pdf)} pages, showing all of them (the answer, Art. 13.4, is on page {answer_page + 1})")
for page_number in range(len(pdf)):
    print(f"--- page {page_number + 1} ---")
    if page_number == answer_page:
        print(">>> This is the page with the answer (Art. 13.4) <<<")
    display(pdf[page_number].render(scale=1.2).to_pil())

## 2. Extracting the text

An LLM cannot read a PDF file like we do, it needs **text**. Turning a PDF into text is called *extraction*, and everything downstream depends on its quality. We use [`pypdf`](https://pypdf.readthedocs.io/), a simple library. (Scanned PDFs, tables and columns might need more than that.)

In [ ]:
reader = PdfReader(PDF_PATH)
text = "\n".join(page.extract_text() for page in reader.pages)

print(f"{len(reader.pages)} pages, {len(text)} characters\n")
print(text[:1500])

Extraction can be tricky, so do not underestimate this step.

Here the document is simple and the text is readable, but with some imperfections: words broken at the end of a line (`pro -\nvides`), page footers in the middle of the text, and a table of contents at the top that repeats all the article titles. For our purposes this is good enough.

We do three cleanups: cut the table of contents (the body starts at the *second* "Art. 1"), join the words broken at the end of a line, and remove the page footers. This will lead to higher quality chunks, and thus answers.

In [ ]:
START = "Art. 1 Individual insurance contract"
first = text.index(START)
document = text[text.index(START, first + 1) :]
document = re.sub(r"(?<=\w) ?-\n(?=\w)", "", document)  # "pro -\nvides" -> "provides"
document = re.sub(r"(?m)^(CAGA04-E2 – Edition|Company under|Administrative details|The insurer is Groupe).*\n", "", document)  # page footers

print(f"{len(document)} characters\n")
print(document[:700])

## 3. Without RAG

### 3.1 A model without context

Let's ask the question from section 1. Only the document can answer it: it is a **specific rule of this insurer**, written in **one article**, so a model is unlikely to know it. It is not general knowledge.

Expected answer (see section 1): yes, within **10 days** of becoming aware of the payment (Art. 13.4), and the insurance ends 14 days after the notice is sent.

In [ ]:
question = "Groupe Mutuel just paid out a claim under my supplementary health insurance. Can I cancel this supplementary insurance because of it, and how many days do I have?"

t0 = time.perf_counter()
response_direct = client.responses.create(model=CHAT_MODEL, input=question)
time_direct = time.perf_counter() - t0

print(response_direct.output_text)
print(f"\nInput tokens: {response_direct.usage.input_tokens}, output tokens: {response_direct.usage.output_tokens}, time: {time_direct:.1f} s")

Read the answer above carefully. A model without our document answers from its **general knowledge**. Depending on the run, you may get a generic reply ("it depends on your contract") or a fluent, confident and precise-sounding one, possibly naming a law or a number of days. In our run, for example, the answer sounded very sure, but it did not give the rule of our document (**10 days**, Art. 13.4).

Whatever the wording, two things hold: the answer **cannot be checked**, because no source is given, and it **may be wrong without any sign of it**. A plausible answer that is not grounded in our document is what we call a **hallucination**.

*Run the cell a few times: the wording changes, the lack of a source does not.*

### 3.2 A model with the *whole text* in the prompt

The simplest fix: paste the document into the prompt. Note the structure of the prompt: **instructions**, then the **document**, then the **question**. We will reuse this shape later.

In [ ]:
prompt_full = f"""Answer the question using only the document below.

DOCUMENT
{document}

QUESTION
{question}
"""

t0 = time.perf_counter()
response_full = client.responses.create(model=CHAT_MODEL, input=prompt_full)
time_full = time.perf_counter() - t0

print(response_full.output_text)
print(f"\nInput tokens: {response_full.usage.input_tokens}, output tokens: {response_full.usage.output_tokens}, time: {time_full:.1f} s")

This works, and for **one document of 9 pages** it is fine. But look at the input tokens: we sent the *whole* document to answer a question about *one* article, and we would do it again for every question. This does not scale, for the reasons given in the introduction (context size, cost and, sometimes, quality).

### 3.3 Fine-tuning a model

The third option is to fine-tune a model on our documents. As explained in the introduction, this is not the way to go here: it is complex, the model is frozen once trained, and it gives no sources.

## 4. RAG

RAG consists of two pipelines. The first one is run **once** (and again when the documents change), the second one **for every question**.

```
 INDEXING  (offline, once)
 ─────────────────────────────────────────────────────────────────────
  documents ──► 1. chunk ──► 2. embed each chunk ──► 3. store in a
                 (pieces)     (text → numbers)        vector store


 QUERYING  (online, every question)
 ─────────────────────────────────────────────────────────────────────
  question ──► 4. retrieve ──► 5. build the prompt ──► 6. LLM answers
               embed the question, (instructions +      (generation)
               take the k most     chunks + question)
               similar chunks
```

The word **retrieval** (step 4) is the new part. Steps 5 and 6 are what we did in 3.2, except that we paste only the retrieved chunks instead of everything.

Let's build each step.


## 5. Step 1: Chunking

We do not embed the whole document as *one* vector: the meaning of a long text gets diluted, and we would be pasting the whole document into the prompt again. Instead we cut it into **chunks**, small pieces that each cover one idea (i.e. self-contained). A chunk is the unit that we retrieve and paste into the prompt.

**Why chunk at all?**

- **For the LLM:** we want to paste only the relevant passages into the prompt, not the whole document: fewer tokens means lower cost, lower latency, and less unrelated text for the model to read through.
- **For the embedding model:** it accepts only a limited input length, and a long text is squeezed into a single vector of fixed size, so each idea gets only a small share of it.
- **For retrieval:** one vector per chunk (one idea) is a more precise target for a question than one vector per document.
- **For citations:** a small chunk points to the exact place the answer comes from.

### 5.1 The simplest way: fixed-size windows

Cut every *N* characters. Easy, but blind to the content. Let's see what it does here.

In [ ]:
def chunk_fixed(text: str, size: int = 400, overlap: int = 100) -> list[str]:
    """Cut a text every `size` characters, each window overlapping the previous one by `overlap` characters."""
    return [text[i : i + size] for i in range(0, len(text), size - overlap)]


fixed_chunks = chunk_fixed(document)
print(f"{len(fixed_chunks)} chunks\n")

for i, chunk in enumerate(fixed_chunks[2:4], start=2):
    print(f"--- chunk {i} ---")
    print(chunk)

Look at where the chunks start and end: they cut sentences in half, and a chunk can begin in the middle of one article and end in the next. Such chunks are hard for an LLM to use, and harder to retrieve.

### 5.2 Better: cut along the structure of the document

Legal texts are organised in **articles** (`Art. 3 Definitions`). We cut at every article heading, so each chunk is one article, **with its heading**. The heading gives every chunk its own context (which also helps the embedding in the next step) and gives us what we need for citations.

For the same reason, each chunk also remembers its **page** in the PDF. Later, a citation like *[Art. 13.4, p. 3]* lets the reader open the exact page and check the answer.

In [ ]:
ARTICLE_HEADING = re.compile(r"(?m)^Art\. (\d+[a-z]?) +[A-Z]")  # at the start of a line: "Art. 3 Benefits"

# Text of each page (with the same line-end fix as in section 2), to find the page of an article
pages_text = [re.sub(r"(?<=\w) ?-\n(?=\w)", "", page.extract_text()) for page in reader.pages]


def find_page(heading: str) -> int:
    """Page number (starting at 1) of an article heading. The table of contents on the first page lists
    the heading too, so we take the last page that contains it."""
    return max(n for n, page_text in enumerate(pages_text, start=1) if heading in page_text)


def chunk_by_articles(text: str) -> list[dict]:
    """One chunk per article. Each chunk keeps its heading, and remembers its article and its page."""
    starts = [m.start() for m in ARTICLE_HEADING.finditer(text)]
    chunks = []
    for begin, end in zip(starts, starts[1:] + [len(text)], strict=True):
        piece = text[begin:end].strip()
        label = "Art. " + ARTICLE_HEADING.match(piece).group(1)  # "Art. 13"
        title = piece.split("\n")[0].strip()  # first line of the heading
        chunks.append({"article": label, "title": title, "page": find_page(title), "text": piece})
    return chunks


chunks = chunk_by_articles(document)

print(f"{len(chunks)} chunks\n")
for i, c in enumerate(chunks):
    print(f"[{i:2d}] {c['title']:50s} page {c['page']}  {len(c['text']):5d} characters")

print("\nExample chunk:\n")
print(next(c for c in chunks if c["article"] == "Art. 12")["text"])

Now each chunk is self-contained, and we can cite it.

There is no universally best chunk size or method. Small chunks are precise but may lack context. Large chunks keep the context but are diluted and cost more in the prompt. Splitting along the structure of the document (articles, headings, paragraphs) is usually a better start than fixed-size windows.

*Bonus; Other techniques*:

- **Chunk overlap**: neighbouring chunks share some text (like the `overlap` parameter above), so that a sentence or an idea cut at a border still appears whole in one of the two chunks. Useful for fixed-size windows, and also to split a **long article** (here Art. 42 *Data protection* is much longer than the others) into pieces with a bit of shared context.
- **Recursive splitting**: try the biggest separator first (article, then paragraph, then sentence) and only cut further when a piece is still too big.
- **Semantic chunking**: cut where the *meaning* changes, detected with embeddings.
- **Context extension (parent-document retrieval, or "small-to-big")**: retrievers prefer **small chunks** (a precise embedding of one idea), but LLMs prefer **wider context**. So we search on small chunks and paste a larger piece into the prompt: the whole parent article or section, or the neighbouring chunks. It is cheap to implement: store the id of the parent in the metadata.
- **Add context to each chunk**: for example the title of the document or a one-line summary, so that a chunk stays understandable on its own.

## 6. Step 2: Embeddings

How do we find the chunks that are relevant for a question? Searching for the *words* of the question fails quickly: the customer writes "Can I walk away from the policy after you paid me?" and the text says "the policyholder has the right to withdraw from the insurance product". Same meaning, few shared words.

An **embedding model** solves this. It turns a text into a **vector**, a list of numbers (here: 1536 of them), such that texts with a similar *meaning* end up as vectors pointing in a similar *direction*.

In [ ]:
embedding_tokens = 0  # running total of the tokens sent to the embedding model


def embed(texts: list[str]) -> np.ndarray:
    """Embed a list of texts. Returns an array of shape (number of texts, dimensions), one row per text."""
    global embedding_tokens
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    embedding_tokens += response.usage.total_tokens
    vectors = np.array([item.embedding for item in response.data])
    # Scale every vector to length 1. OpenAI already does this, other servers (e.g. Ollama) may not.
    return vectors / np.linalg.norm(vectors, axis=1, keepdims=True)


vector = embed(["The insurance term is one calendar year running from 1 January to 31 December."])[0]

print("Dimensions:", vector.shape[0])
print("First 8 values:", np.round(vector[:8], 4))
print("Length of the vector:", np.linalg.norm(vector).round(4))

The numbers themselves mean nothing to us. What matters is the **distance between vectors**. The standard measure is the **cosine similarity**: it compares the *direction* of two vectors.

- Close to `1` = very similar meaning, close to `0` = unrelated, `-1` = opposite.
- Since all our vectors have length 1, it is simply the **dot product** of the two vectors (multiply the numbers pairwise and add them up). You do not need the maths: *higher score = closer meaning*.

Let's test it with a question and four sentences: three from our document, one unrelated.

In [ ]:
query = "The insurer has just reimbursed me. Can I walk away from the policy?"

sentences = [
    "After each claim for which a benefit is paid by the insurer, the policyholder has the right to withdraw from the insurance product concerned within 10 days.",
    "The insured person must report accidents to the insurer within 10 days of their occurrence.",
    "Premiums are payable yearly in advance in Switzerland.",
    "The Matterhorn is located in the canton of Valais.",
]

query_vector = embed([query])[0]
sentence_vectors = embed(sentences)

for sentence, vec in zip(sentences, sentence_vectors, strict=True):
    print(f"{np.dot(query_vector, vec):.3f}  {sentence}")

The withdrawal sentence wins, although it shares almost no words with the question ("reimbursed" and "walk away from the policy" versus "benefit is paid" and "withdraw from the insurance product"). This is the strength of embeddings: they match **meaning, not vocabulary**.

Note that the sentence about accidents (also "10 days", also about the insurer) comes second: it is *close* in topic, but not what we need. 

**Embeddings can return passages that are related but not right.** This is why we do not simply show the retrieved text to the user: we give it to the LLM, which reads the chunks, keeps the relevant ones and writes a prose answer.

*Bonus*: The scores are **relative, not calibrated**: they rank chunks *for one question*, but they do not tell whether the best chunk really answers it. A question that the document cannot answer can get a top score as high as (or higher than) one it can, because the chunks are still about insurance, premiums, claims... There is no universal threshold that separates "found" from "not found", so we cannot filter on the score alone. The instruction in the prompt ("say you cannot find it") does that job.

## 7. Step 3: The vector store

Now we embed **all** chunks, once, and keep them in a **vector store**. For every chunk it stores three things: the **text**, its **embedding**, and some **metadata** (here the article and the page, for citations). Given the embedding of a question, it returns the stored chunks whose embedding is closest.

We use a minimal vector store (the class below, a few lines of Python, no magic). A production vector database (e.g. [Chroma](https://docs.trychroma.com/), which the Auto-Quote backend uses, or pgvector) does the same, plus fast approximate search for millions of chunks, persistence on disk and filtering.

In [ ]:
class MiniVectorStore:
    """Keeps (embedding, text, metadata) lists and returns the ones closest to a query's embedding."""

    def __init__(self):
        self.embeddings: list[np.ndarray] = []
        self.texts: list[str] = []
        self.metadata: list[dict] = []

    def add(self, embeddings, texts: list[str], metadata: list[dict] | None = None) -> None:
        if metadata is None:
            metadata = [{} for _ in texts]
        assert len(embeddings) == len(texts) == len(metadata)
        self.embeddings += [np.asarray(e) for e in embeddings]
        self.texts += list(texts)
        self.metadata += list(metadata)

    def query(self, embedding, k: int = 3, keep=None) -> list[dict]:
        """The k entries most similar to `embedding`, best first.

        keep: optional filter on the metadata; only entries where keep(metadata) is True are ranked.
        Each result is {"index", "score", "text", "metadata"}.
        """
        candidates = [i for i in range(len(self.texts)) if keep is None or keep(self.metadata[i])]
        # vectors have length 1, so the cosine similarity is the dot product
        scored = sorted(((float(np.dot(embedding, self.embeddings[i])), i) for i in candidates), reverse=True)
        return [
            {"index": i, "score": score, "text": self.texts[i], "metadata": self.metadata[i]}
            for score, i in scored[:k]
        ]

    def __len__(self) -> int:
        return len(self.texts)


store = MiniVectorStore()
tokens_before = embedding_tokens
store.add(
    embeddings=embed([c["text"] for c in chunks]),  # we compute the embeddings ourselves
    texts=[c["text"] for c in chunks],
    metadata=[{"article": c["article"], "title": c["title"], "page": c["page"]} for c in chunks],
)

print(len(store), "chunks stored")
indexing_tokens = embedding_tokens - tokens_before
print(f"Indexing cost: {indexing_tokens} tokens, about USD {indexing_tokens * 0.02 / 1e6:.6f} (text-embedding-3-small: USD 0.02 per million tokens)")

In [ ]:
# What is stored for one chunk: its text, its embedding and its metadata
i = 12
print("text:     ", " ".join(store.texts[i].split())[:70], "...")
print("embedding:", np.round(store.embeddings[i][:5], 3), "... (1536 numbers)")
print("metadata: ", store.metadata[i])

## 8. Step 4: Retrieval

To retrieve, we embed the question with the **same model** and ask the store for the `k` most similar chunks (the *top-k*), with their score.

In [ ]:
def retrieve(question: str, k: int = 3) -> list[dict]:
    """The k chunks most similar to the question, best first."""
    hits = store.query(embed([question])[0], k=k)
    return [
        {"article": h["metadata"]["article"], "title": h["metadata"]["title"], "page": h["metadata"]["page"], "text": h["text"], "score": h["score"]}
        for h in hits
    ]


results = retrieve(question, k=3)

for rank, r in enumerate(results, start=1):
    print(f"[{rank}] score {r['score']:.3f}  {r['title']}  (page {r['page']})")
    print("    " + " ".join(r["text"].split())[:150] + "...")

The question is about cancelling after a payment, so the article we are hoping to see is **Art. 13 Duration and termination of insurance contract**. Check where it lands in the list above. In our run, for example, it was in the top 3 but **not first**, and the three scores were almost the same.

Two things to keep in mind:

- Retrieval is **approximate**: it finds the right *neighbourhood*, not always the exact best chunk. That is why we keep the top *k* chunks (here 3) and not just one. As long as the right article is among them, the LLM can pick the relevant one.
- Scores can be close together, much closer than in the toy example of section 6, because here we compare the question with **whole articles** (long, covering several topics) instead of one clean sentence. Long articles also add many tokens to the prompt, whether they help or not.

*Bonus; Hybrid retrieval.* Embeddings match meaning, but they are weak on **exact strings**: product codes, article numbers, names, acronyms. A customer who writes *"What does CAGA04-E2 say about termination?"* or *"RFBA02 admission"* needs a **keyword search** (e.g. BM25, like a classic search engine) that finds exactly that code, because an embedding of "CAGA04-E2" carries almost no meaning. The usual answer is **hybrid retrieval**: run both searches (embeddings + keywords) and merge the two ranked lists (for example with *reciprocal rank fusion*). Many vector stores (and Elasticsearch, OpenSearch...) offer it out of the box. The deep-dive notebook explores it.

## 9. Step 5: The augmented prompt

We now build the prompt: instructions, the retrieved chunks, and the question. Three details make a difference:

- **"Use only the extracts below"** pushes the model to answer from the document rather than from its general knowledge.
- **Citations**: each chunk carries its article heading and its page, so the model can say where every fact comes from, for example *[Art. 13.4, p. 3]*.
- **A way out:** if the extracts do not contain the answer, the model must say so, instead of inventing something.

In [ ]:
PROMPT_TEMPLATE = """Answer the question using only the extracts below.
Cite the article and the page of every fact in square brackets, e.g. [Art. 3, p. 2].
If the text shows a paragraph number, add it to the article, e.g. [Art. 13.4, p. 3].
If the extracts do not contain the answer, say "I cannot find this in the documentation." Do not guess.

EXTRACTS
{context}

QUESTION
{question}
"""


def build_prompt(question: str, results: list[dict]) -> str:
    # every chunk starts with its article heading; we add its page just before
    context = "\n\n".join(f"[Page {r['page']}]\n{r['text']}" for r in results)
    return PROMPT_TEMPLATE.format(context=context, question=question)


prompt_rag = build_prompt(question, results)
print(prompt_rag)

## 10. Step 6: Generation

The last step is the call we already know. The model now sees only the relevant articles.

In [ ]:
t0 = time.perf_counter()
response_rag = client.responses.create(model=CHAT_MODEL, input=prompt_rag)
time_rag = time.perf_counter() - t0

print(response_rag.output_text)

The answer should be grounded in the document and cite the article and the page, so a human can check it, even by opening the page shown above. Compare it with 3.1: this time the figures come from the document, not from the model's general knowledge. In our run, the answer gave the 10-day rule with its citation.

Keep in mind that the model **paraphrases**, so the wording can differ slightly from the article (for example "sent" vs "received"). That is exactly why the citation is there: to let you check the exact wording.

Let's wrap retrieval, prompt and generation in one function, and try a few more questions, including ones that the document **cannot** answer.

In [ ]:
def ask(question: str, k: int = 3) -> None:
    results = retrieve(question, k=k)
    response = client.responses.create(model=CHAT_MODEL, input=build_prompt(question, results))

    print(f"Q: {question}\n")
    print("Retrieved:", [f"{r['article']}, p. {r['page']} ({r['score']:.2f})" for r in results])
    print(f"\nA: {response.output_text}")
    print("\n" + "=" * 90 + "\n")


ask("I am in the hospital since yesterday. By when must I tell the insurer?")  # expected: within 5 days of admission, before admission if coverage must be guaranteed (Art. 31.2)
ask("Who pays the doctor, me or the insurer?")  # expected: you pay the provider, then claim reimbursement; exception: direct payment (Art. 17.1)
ask("Is an illness that I already had before signing covered?")  # expected: excluded (Art. 18.1.a, see also Art. 8). Does retrieval find Art. 18?
ask("What is the monthly premium for a 40-year-old living in Zurich?")  # not in this document!
ask("Does Groupe Mutuel cover LASIK laser eye surgery?")  # not in this document either!

Compare each answer with the expectation in the comments, and look at the **Retrieved** line each time. Is the article we expected in the list?

- For the first two questions, you should get an answer with a citation that can be checked.
- For the third, the answer may be only partial if the expected article (Art. 18) was not retrieved. In our run, for example, it was not, and the answer relied on other articles. This is a retrieval miss, and the retrieved chunks tell you why.
- The last two questions **cannot be answered from this document**, yet retrieval still returns three chunks each time (it always does). The instruction in the prompt tells the model to say "I cannot find this in the documentation". In our run it did, though it sometimes added related context. Without that instruction, the model could have built an answer from loosely related chunks.

## 11. Cost and speed

How does RAG compare with the two other ways we tried: the **direct LLM call** (no document, 3.1) and the **whole document in the prompt** (3.2)? The direct call has the smallest prompt, but it cannot answer. So the real question is: how much does RAG add on top of it, and how much does it save compared with the whole document?

### 11.1 Cost

Where does RAG cost money?

- **Indexing:** one embedding call for the chunks, once. Embeddings are very cheap (`text-embedding-3-small`: about USD 0.02 per 1 million tokens).
- **Every question:** one embedding call for the question (negligible), plus the generation, priced per input and output token.

The important term is the **input of the generation**. Compare the three calls we made:

In [ ]:
PRICE_PER_MILLION_INPUT_TOKENS = 0.10  # USD, gpt-6-luna. Check the provider's price page for real budgets.

runs = [("Direct call, no context", response_direct), ("Whole document in prompt", response_full), ("RAG, top-3 chunks", response_rag)]

for name, response in runs:
    tokens = response.usage.input_tokens
    cost = tokens * PRICE_PER_MILLION_INPUT_TOKENS / 1e6
    print(f"{name:26s} {tokens:6d} input tokens   {response.usage.output_tokens:5d} output tokens   input cost USD {cost:.6f} per question")

saved = 1 - response_rag.usage.input_tokens / response_full.usage.input_tokens
print()
print(f"RAG sends {100 * saved:.0f}% fewer input tokens than the whole document.")

Compare the **input** column first. The direct call has the smallest input, since it only contains the question, but it cannot answer from our document. The RAG call sends only a fraction of the whole-document prompt (in our run, around a quarter).

The **output** column can differ a lot between calls, and not only because of the length of the visible answer: some models also "think" before answering, and those tokens are billed as output. In our run, the direct call produced many more output tokens than the other two, even though its visible answer was short. This depends on the model and changes from run to run, so the comparison that matters here is the **input**, which is the part RAG controls.

RAG adds **a few chunks** (about 3 articles) and one embedding call: a small overhead. The whole-document prompt grows with the size of the knowledge base. The RAG prompt does not (always `k` chunks).

### 11.2 Speed

Retrieval adds a small step (one embedding call plus a search), which should take a fraction of a second. Fewer input tokens usually means a faster answer too, but **on a single 9-page document the gap can be small**. In our run, the whole-document call and the RAG call took roughly the same time. The gap grows with the size of the knowledge base.

Speed also depends on the **length of the answer** and on any internal reasoning of the model, so the direct call, despite its small prompt, can be the slowest. Timings are noisy: run the cells a few times before drawing conclusions.

In [ ]:
t0 = time.perf_counter()
results = retrieve(question, k=3)
time_retrieval = time.perf_counter() - t0

print(f"Direct call, no context:  {time_direct:.1f} s")
print(f"Whole document in prompt: {time_full:.1f} s")
print(f"RAG: retrieval {time_retrieval:.2f} s + generation {time_rag:.1f} s = {time_retrieval + time_rag:.1f} s")

## 12. Takeaways

- **RAG grounds the LLM's answers in your own documents**, and lets it **cite** them (article, page) so a human can check.
- **Two pipelines:** *indexing* (offline: documents → chunks → embeddings → vector store) and *querying* (online: question → retrieval → augmented prompt → answer).
- **Data quality is now the main bottleneck**, more than the LLM: extraction and chunking decide what *can* be found.
- **Chunking, embeddings, retrieval and the prompt are the levers you can tune.** Embeddings match meaning, not exact strings (codes, names): add keyword search if needed. Retrieval always returns *k* chunks, so tell the model what to do when the answer is not among them.
- **When an answer is wrong, look at the retrieved chunks first:** is it a retrieval problem or a generation problem?
- **RAG vs. pasting everything:** pasting is fine for one short document. RAG pays off when the knowledge base is large or changes often, when cost matters, or when answers must be traceable.
- **RAG is evolving and is large** (agentic RAG, multimodal RAG, GraphRAG, Re-ranker, metadata filtering...).

## 13. *Bonus:* Agentic RAG

Instead of one fixed pass (retrieve once, then answer), an LLM *agent* controls the process and uses retrieval as a **tool** it can call. It can:
  - **decide not to retrieve** (a greeting, a general question),
  - **rewrite the query** and **retrieve again** if the first chunks are not good enough,
  - **call retrieval several times** (one question per topic, or to follow a reference such as "the special terms prevail", Art. 39),
  - pick **which source** to search (general terms, special terms, the customer's contract),
  - check its answer before replying.

A typical case: "Does my supplemental insurance cover dental treatment?" needs the product's own terms, the Dentaire Plus terms *and* the general terms, and one retrieval pass rarely finds all three. Agentic RAG is more powerful for complex questions, at the price of more LLM calls: more cost, more latency, and less predictable behaviour.


## 14. *Bonus:* How can RAG go wrong?

RAG is a chain, and every link can break. When an answer is bad, find the link:

| Where | What goes wrong | Example here |
|---|---|---|
| **Extraction** | Text is garbled, tables or columns are scrambled, scanned pages give nothing. Everything downstream is wrong. | Words cut at line ends, footers inside articles. |
| **Chunking** | Chunks cut an idea in half, are too big (diluted) or too small (no context). | Fixed windows that start in the middle of a sentence. |
| **Retrieval: chunk not found** | The right chunk is not in the top-k: vocabulary mismatch, vague question, or an exact code that embeddings do not capture. | The "nose job" question (section 16). Fixes: query expansion, hybrid retrieval, a larger `k`, a reranker. |
| **Retrieval: wrong chunk found** | A chunk that is similar but not relevant ranks first. | The accident-report article (10 days) is close to the withdrawal article (10 days). |
| **Generation: chunk ignored** | The right chunk is in the prompt, but the model answers from its own knowledge, misreads it, or misses a detail (more likely when the prompt is long or full of similar passages). | The model quotes a generic "14 days" instead of the 10 days of Art. 13.4. |
| **Generation: no way out** | The prompt has no instruction to say "I cannot find this", so the model invents an answer when nothing relevant was retrieved. | The premium question for a 40-year-old. |
| **Versioning** | Several versions of a document are indexed: old and new conditions give conflicting answers, or an outdated chunk is cited. | Two editions of the same terms in the folder (see the conflict exercise in the deep-dive notebook). Fix: metadata (edition, date), filter, re-index on change. |
| **Stale index** | The documents changed (new article, corrected number) but the index was never rebuilt. Answers look fine and cite an article, but the content is out of date, and nothing in the pipeline warns about it. | A rule is changed in the PDF, but the store still holds the old text. Fix: re-index on every change, keep a document date or hash in the metadata and show it in the citation. |
| **Missing context** | The answer needs another document: for example the *special* terms prevail over these general terms (Art. 39), but they were not retrieved. | A question about one product needs its special terms and the general terms. |

Note that most of the time, the challenge is in the data extraction, chunking and retrieval strategy.


## 15. *Bonus*: How do we know if it works? Three metrics

Do not just eyeball the answers: measure on a set of test questions. Here are three classic checks. Each looks at one link of the chain, and they are often scored by a second LLM acting as a judge, or by a human:

| Metric | Question it answers | Compares | Good / bad example |
|---|---|---|---|
| **Context relevance** | Are the retrieved chunks relevant to the question? | question ↔ chunks | *Good:* "Can I cancel after a payment?" retrieves Art. 13. *Bad:* the nose-job question retrieves Art. 7, 28 and 36. |
| **Groundedness** (faithfulness) | Is every claim of the answer supported by the chunks? | answer ↔ chunks | *Good:* "10 days" and Art. 13.4 says 10 days. *Bad:* "you also get a refund of the premium" while no retrieved chunk says that. |
| **Answer relevance** | Does the answer actually address the question? | answer ↔ question | *Good:* says yes, and gives the number of days. *Bad:* an accurate paragraph on the statute of limitations (Art. 38): grounded, but not what was asked. |

Read together, they tell you where to look:

- low context relevance: fix retrieval or chunking,
- low groundedness: fix the prompt or the model,
- low answer relevance: the model drifted away from the question.

A related metric, when you have reference answers, is **context recall**: was the chunk that holds the answer retrieved at all?

## 16. *Bonus*: query expansion

The customer who asks the question and the people who wrote the document use different words. A customer writes *"Will the insurance pay for my nose job?"*, while the contract talks about *"operations designed to correct physical defects or cosmetic imperfections of an aesthetic nature"*. Embeddings help, but retrieval can still miss. Let's see:

In [ ]:
question_vague = "Will the insurance pay for my nose job? I just don't like how it looks."

print("Expected: Art. 18 Exclusions (point 1.f: cosmetic operations are excluded)\n")
for rank, r in enumerate(retrieve(question_vague, k=3), start=1):
    print(f"[{rank}] score {r['score']:.3f}  {r['title']}")

Art. 18 is not among the top-3 chunks: the question is short, colloquial, and contains no word of the article. The model will not see the exclusion, so it cannot answer correctly, no matter how good the prompt is.

**Query rewriting** fixes this on the question side: we ask an LLM to restate the question in the **formal vocabulary of the documents**, and we search with that instead.

In [ ]:
REWRITE_PROMPT = """Rewrite the customer question below as one search query for the general terms and conditions of a health insurance contract.
Use the formal vocabulary such a contract would use (e.g. "exclusions", "termination", "benefits").
Output only the query.

QUESTION
{question}
"""


def rewrite_query(question: str) -> str:
    """Ask the LLM for one search query that restates the question."""
    return client.responses.create(model=CHAT_MODEL, input=REWRITE_PROMPT.format(question=question)).output_text.strip()


rewritten = rewrite_query(question_vague)
print("Rewritten:", rewritten)
print()

for rank, r in enumerate(retrieve(rewritten, k=3), start=1):
    print(f"[{rank}] score {r['score']:.3f}  {r['title']}")

The rewritten query speaks the language of the contract ("rhinoplasty", "cosmetic", "exclusions"), and **Art. 18 Exclusions now appears in the top 3**. The exact scores and ranks change from run to run, but the effect is the same: restating the question in the vocabulary of the document helps retrieval a lot.



**Query expansion** goes one step further: the LLM writes **several** search queries, covering different aspects of the question. We retrieve for the original question *and* every rewrite, then merge the results (we keep the best score of each chunk). One query can miss, several queries are more likely to hit.

In [ ]:
EXPANSION_PROMPT = """Rewrite the customer question below as {n} different search queries for the general terms and conditions of a health insurance contract.
Use the formal vocabulary such a contract would use (e.g. "exclusions", "termination", "benefits") and cover different aspects of the question.
One query per line, no numbering, no other text.

QUESTION
{question}
"""


def expand_query(question: str, n: int = 3) -> list[str]:
    """Ask the LLM for n search queries that rephrase the question."""
    response = client.responses.create(model=CHAT_MODEL, input=EXPANSION_PROMPT.format(n=n, question=question))
    return [line.strip() for line in response.output_text.splitlines() if line.strip()][:n]


def retrieve_expanded(question: str, k: int = 3, n: int = 3) -> list[dict]:
    """Retrieve for the question and for its rewrites, merge, keep the best k chunks."""
    queries = [question, *expand_query(question, n)]
    print("Queries:")
    for q in queries:
        print("  -", q)
    best = {}  # article -> best result over all queries
    for q in queries:
        for r in retrieve(q, k=k):
            if r["article"] not in best or r["score"] > best[r["article"]]["score"]:
                best[r["article"]] = r
    return sorted(best.values(), key=lambda r: r["score"], reverse=True)[:k]


results_expanded = retrieve_expanded(question_vague)

print("\nRetrieved:")
for rank, r in enumerate(results_expanded, start=1):
    print(f"[{rank}] score {r['score']:.3f}  {r['title']}")

Now the answer, without and with query expansion (same prompt, same `k`):

In [ ]:
for name, retrieved in [("Plain retrieval", retrieve(question_vague, k=3)), ("With query expansion", results_expanded)]:
    answer = client.responses.create(model=CHAT_MODEL, input=build_prompt(question_vague, retrieved)).output_text
    print(f"### {name}: {[r['article'] for r in retrieved]}\n{answer}\n")

The price: one **extra LLM call** (a small one) and a few more embedding calls, so a bit more cost and latency before the answer starts. It pays off for short, vague or colloquial questions, and is a waste for questions that already use the document's words.

Variants:

- **HyDE** (*Hypothetical Document Embeddings*): ask the LLM to write a **hypothetical answer** to the question, even without knowing the document, and search with *that* text. An answer looks more like the document passages than a question does.
- **Sub-questions**: split a complex question into several simple ones, with one retrieval each.
- **Step-back**: also ask a more general version of the question.